<a href="https://colab.research.google.com/github/tsukki8/ds2002-fa26/blob/main/notebooks/07-public-api/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [3]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed

    last_error = None

    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            error = f"HTTP {r.status_code}: {r.text[:200]}"

            # Invalid requests should fail immediately.
            if r.status_code in (400, 404):
                raise RuntimeError(error)

            # Other non-retryable statuses should also fail.
            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(error)

            last_error = error

        except requests.RequestException as e:
            # Network errors may be transient.
            last_error = str(e)

        if attempt < tries - 1:
            delay = 0.5 * (2 ** attempt)
            print(f"Attempt {attempt + 1} failed: {last_error}")
            print(f"Retrying in {delay:.1f} seconds...")
            time.sleep(delay)

    raise RuntimeError(
        f"Weather request failed after {tries} attempts: {last_error}"
    )

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [6]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False

try:
    weather_raw = get_weather('2026-09-05', '2026-10-03')

    weather_raw['time'] = pd.to_datetime(weather_raw['time'])

    target_dates = pd.to_datetime([
        '2026-09-05',
        '2026-10-03'
    ])

    wx = weather_raw[
        weather_raw['time'].isin(target_dates)
    ].copy()

    # Fall back if the request succeeded but did not return both dates.
    if len(wx) != 2:
        raise ValueError(
            f"Expected 2 target dates, but received {len(wx)}"
        )

except Exception as e:
    print(f"Weather fetch failed: {e}")
    used_fallback = True

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [11]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
if used_fallback:
    wx = FALLBACK.copy()
    wx['time'] = pd.to_datetime(wx['time'])
    wx['source'] = 'fallback sample'
else:
    wx['source'] = 'open-meteo'

print("Data source:", wx['source'].iloc[0])
print(wx)

Data source: open-meteo
         date  temp_max  precip_mm      source
0  2026-09-05      29.4        2.5  open-meteo
28 2026-10-03      22.0       14.7  open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [12]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

print(wx)

         date  temp_max  precip_mm      source
0  2026-09-05      29.4        2.5  open-meteo
28 2026-10-03      22.0       14.7  open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [13]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [14]:
# TODO
sept = wx.loc[
    wx['date'] == pd.Timestamp('2026-09-05'),
    'precip_mm'
].iloc[0]

october = wx.loc[
    wx['date'] == pd.Timestamp('2026-10-03'),
    'precip_mm'
].iloc[0]

difference = abs(sept - october)

if sept > october:
    wetter_date = 'September 5, 2026'
elif october > sept:
    wetter_date = 'October 3, 2026'
else:
    wetter_date = None

if wetter_date is None:
    print("Both dates had the same precipitation: "
          f"{sept:.1f} mm.")
else:
    print(f"{wetter_date} was wetter by "
          f"{difference:.1f} mm of precipitation.")

October 3, 2026 was wetter by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [15]:
# TODO
from datetime import datetime

print("Source: Open-Meteo Historical Weather API")
print("Endpoint:", BASE)
print("Latitude: 38.03")
print("Longitude: -78.51")
print("Date range: 2026-09-05 to 2026-10-03")
print("Pulled at:", datetime.now().astimezone().isoformat())
print("Data type:", wx['source'].iloc[0])

Source: Open-Meteo Historical Weather API
Endpoint: https://archive-api.open-meteo.com/v1/archive
Latitude: 38.03
Longitude: -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-09T02:04:01.320235+00:00
Data type: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame on failure instead of raising, merging get_weather with 200 days of sales would leave the sales rows without matching weather data. The chart would show a blank precipitation vs. revenue relation or ignore weather-related patterns. It would seem like no useful relationship occured between rain and revenue. No error would occur after the merge, so missing data could be interpreted as a real finding and no one would catch it.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [16]:
# stretch
cities = {
    'Charlottesville': (38.03, -78.51),
    'Orlando': (28.54, -81.38),
}

frames = []

for city, (lat, lon) in cities.items():
    try:
        city_weather = get_weather(
            '2026-09-05',
            '2026-10-03',
            lat=lat,
            lon=lon
        )

        city_weather['date'] = pd.to_datetime(
            city_weather['time']
        )
        city_weather['city'] = city
        city_weather['source'] = 'open-meteo'

        city_weather = city_weather.rename(columns={
            'temperature_2m_max': 'temp_max',
            'precipitation_sum': 'precip_mm'
        })

        city_weather = city_weather[
            city_weather['date'].isin(
                pd.to_datetime(['2026-09-05', '2026-10-03'])
            )
        ]

        frames.append(
            city_weather[
                ['date', 'city', 'temp_max', 'precip_mm', 'source']
            ]
        )

    except Exception as e:
        print(f"Could not fetch {city}: {e}")

if frames:
    city_weather_all = pd.concat(frames, ignore_index=True)
    print(city_weather_all)

        date             city  temp_max  precip_mm      source
0 2026-09-05  Charlottesville      29.4        2.5  open-meteo
1 2026-10-03  Charlottesville      22.0       14.7  open-meteo
2 2026-09-05          Orlando      32.1        3.9  open-meteo
3 2026-10-03          Orlando      30.4       14.5  open-meteo
